# pw02 — 오픈소스 센싱/ISAC 도구 지도

> ⚠ **이 노트북은 생성물이다. 수정은 `prior_work/src/make_pw02.py` 에서** 하고 재실행할 것.
> 모든 사실·라이선스는 `prior_work/outputs/prior_work.json`(직접 GitHub/arXiv 확인) 에서 주입한다.

질문: **이미 존재하는 오픈소스가 우리 조각(패시브 바이스태틱 + SBR+PO 드론 RCS + ECA/CAF/CFAR)을 대신 해 주는가?** 답 — **통째로 해 주는 한 도구는 없지만, 조각마다 검증된 오픈소스가 있다.** 사용자 방침(‘직접 만든 것을 최대한 오픈소스로 대체’)에 맞춰, 각 조각의 대체 도구를 확정한다.

> ⚠️ **1차 조사 정정.** 앞서 '패시브 바이스태틱 ECA 는 드롭인 오픈소스가 없다'고 적었는데, **틀렸다** — **pyAPRiL**(GPLv3, DVB-T/FM 실측 검증)이 정확히 그 드롭인이다(ECA/ECA-B/ECA-S·CAF·CA-CFAR·DoA). 사용자 심화 서베이가 이를 지목했고, GitHub 로 직접 확인했다.

## §1. 능력 매트릭스 (한눈에)

| 도구 | 라이선스 | 담당 계층 | 메쉬 RCS | 검출/처리 | 패시브·바이스태틱 | 채택 |
|---|---|---|---|---|---|---|
| **pyAPRiL** | GPLv3 | 검출체인(ECA/CAF/CFAR) | — | ✅ | ✅ | ⭐ HIGH |
| **RadarSimPy** | GPLv3 | RCS·μD | ✅ | ✅ | — | HIGH |
| **OpenISAC** | 오픈소스 | 실측 SDR | — | ✅ | ✅ | ⭐ HIGH |
| **NIST 5GNRad** | NIST-developed | 5G NR 검출 | — | ✅ | — | HIGH |
| **Stone Soup** | MIT | 추적 | — | — | — | MEDIUM |
| **openEMS** | GPLv3 | full-wave RCS | ✅ | — | — | MEDIUM |
| **GNU Radio** | GPLv3 | 실측 I/O | — | — | — | MEDIUM |
| **NIST ISAC-PLM** | NIST | WiFi(60G) 검출 | — | ✅ | ✅ | LOW-MEDIUM |
| **MATLAB 5G/Radar/Phased Array Toolboxes** | 상용 | 종합 | — | ✅ | — | MEDIUM |
| **OpenAirInterface** | OAI Public License | 실 5G RAN | — | — | — | LOW |
| **ns3sionna** | 오픈소스 | 네트워크 | — | — | — | LOW — 네트워크/MAC 레벨. 우리 물리레이어 패시브 레이더와 층이 다름 |
| **ms-van3t-rt** | 오픈소스 | V2X | — | — | — | LOW — V2X 통신 전용 |

> **읽는 법.** 한 도구가 6칸을 다 채우진 못하지만, **계층마다 대체 도구가 있다** — 검출체인=**pyAPRiL**, RCS=**RadarSimPy/openEMS**, 추적=**Stone Soup**, 실측=**OpenISAC+GNU Radio**. 이게 '직접 만든 것을 오픈소스로 대체'의 실제 지도다(→§4·OPENSOURCE.md).

---
## §2. 채택도 HIGH — 실제로 쓸 것 (우리 코드의 직접 대체/검증)

### 🔎직접확인  pyAPRiL (Advanced Passive Radar Library)  —  채택도 판정: **⭐ HIGH**
- **저장소·출처**: github.com/petotamas/APRiL (pip: pyapril) · BME Radarlab  ([1](https://github.com/petotamas/APRiL) · [2](https://github.com/pyapril/pyapril))
- **라이선스**: GPLv3 (완전 오픈)
- **무엇을 하나**: ⭐ **패시브 레이더 신호처리 전용** — 기준/감시 채널 준비, 기준신호 재생성, Wiener-SMI·**ECA/ECA-B/ECA-S** 클러터제거(공간·시간·공간시간), 상관·주파수영역·배치 overlap-save 검출기, **CAF(교차모호함수)**, 도플러 윈도잉, **CA-CFAR**, plot 추출, 다채널 **DoA**. DVB-T/FM 실측 검증됨
- **표적 산란 처리**: n/a (신호처리 라이브러리 — 채널은 우리가 공급)
- **검출체인?** ✅ **패시브 바이스태틱 검출체인 전체**(ECA→CAF→CFAR) — 우리 passive_process.py 와 1:1 대응
- **우리 프로젝트 채택**: ⭐ HIGH(**드롭인 대체/검증**) — 우리 손으로 만든 ECA/CAF/CFAR 의 **직접 대체 후보**. Sionna 가 만든 r_ref/r_surv 를 넘겨 검출. '검증 후 대체' 원칙: 우리 결과와 대조→일치 시 권위 이관

### 🔎직접확인  RadarSimPy  —  채택도 판정: **HIGH**
- **저장소·출처**: github.com/radarsimx/radarsimpy  ([1](https://github.com/radarsimx/radarsimpy) · [2](https://github.com/radarsimx/radarsimpy/blob/master/LICENSE))
- **라이선스**: GPLv3 (완전 오픈)
- **무엇을 하나**: Python+C++ 레이더 시뮬 — 점표적 + **3D STL 메쉬 RCS**, 마이크로도플러/터빈, Swerling, 거리도플러·DoA·CFAR
- **표적 산란 처리**: 3D 메쉬 RCS 직접 계산(광선추적 기반). 다만 5G/WiFi 통신 파형 스택은 없음
- **검출체인?** ✅ 거리도플러·CFAR·DoA
- **우리 프로젝트 채택**: HIGH(검증 오라클) — 우리 SBR+PO 드론 RCS·프로펠러 마이크로도플러를 **독립 도구로 교차검증**하는 데 이상적. GPLv3 라 자유 사용

### 🔎직접확인  OpenISAC  —  채택도 판정: **⭐ HIGH**
- **저장소·출처**: github.com/zhouzhiwen2000/OpenISAC · arXiv:2601.03535 (2026-01)  ([1](https://arxiv.org/abs/2601.03535) · [2](https://github.com/zhouzhiwen2000/OpenISAC))
- **라이선스**: 오픈소스
- **무엇을 하나**: 실시간 OFDM-ISAC 실험 플랫폼 — 모노+**바이스태틱** 지연도플러, **OTA 동기(유선없이 바이스태틱)**, USRP B200~**X400 계열**, C++PHY+Python센싱, 마이크로도플러 추출 검증
- **표적 산란 처리**: OTA 실측(시뮬 아님)
- **검출체인?** ✅ 지연도플러 검출, 마이크로도플러
- **우리 프로젝트 채택**: ⭐ HIGH(실측 단계) — **X410 지원 + 바이스태틱 OTA 동기**가 사용자 하드웨어 계획과 직결. sim→real 다리. 단 연속 커스텀 OFDM 이라 표준 WiFi/LTE/5G 파형은 별도

### 🔎직접확인  NIST 5GNRad  —  채택도 판정: **HIGH**
- **저장소·출처**: github.com/usnistgov/5GNRad (Steve Blandino, NIST CTL)  ([1](https://github.com/usnistgov/5GNRad))
- **라이선스**: NIST-developed (US Gov, 사실상 퍼블릭 도메인)
- **무엇을 하나**: 5G NR ISAC 링크레벨 시뮬레이터 — 표준호환 NR 파형·채널추정·거리도플러·**CFAR**·각도추정·3D 위치
- **표적 산란 처리**: c) **RCS 점표적** 주입 + target/background 채널 분리(h=h_bg+h_target)
- **검출체인?** ✅ 풀 검출체인(range-Doppler·CFAR·clustering·위치/속도 추정·TP/FP/FN)
- **우리 프로젝트 채택**: HIGH(아키텍처 참조) — h=h_bg+h_target 분리가 **우리와 동일**하다는 강력한 방증. 단 능동/PRS 기반이라 패시브 reference/surveillance 구조는 우리가 추가해야 함

> 🔑 **네 도구의 역할 분담(대체 지도).**
> - **pyAPRiL** (GPLv3) — *검출체인 드롭인*. 우리 손으로 만든 **ECA/CAF/CFAR**(`passive_process.py`)를 **직접 대체**하는 라이브러리. Sionna 가 만든 r_ref/r_surv 를 넘기면 ECA→CAF→CFAR 를 실측 검증된 코드로 수행한다. '검증 후 대체' 원칙: 우리 결과와 대조 → 일치 시 권위 소스를 pyAPRiL 로 이관.
> - **RadarSimPy** (GPLv3) — *RCS 검증 오라클*. 3D 메쉬 RCS·마이크로도플러를 독립 재계산해 SBR+PO(report07/08)와 대조.
> - **OpenISAC** — *실측 다리*. USRP **X410 + 바이스태틱 OTA 동기** — sim→real 골격.
> - **NIST 5GNRad** — *아키텍처 참조*. `h=h_bg+h_target` 가 우리 report12 구조와 동일함을 확인.

> ⚠️ **단, GPU 몬테카를로는 우리 커널 유지.** pyAPRiL 은 NumPy(CPU) 단일실현 중심이라, report12 의 수천 회 배치 MC(K=6000)는 우리 `detection_gpu.py`(torch 배치)가 필요하다. **역할: pyAPRiL=기준/검증 구현·단일실현 분석, 우리 GPU 커널=대량 MC(단 pyAPRiL 로 정합성 검증).**

---
## §3. 채택도 MEDIUM — 특정 단계에서 도입

### 🔎직접확인  Stone Soup  —  채택도 판정: **MEDIUM**
- **저장소·출처**: github.com/dstl/Stone-Soup (英 Dstl)  ([1](https://github.com/dstl/Stone-Soup))
- **라이선스**: MIT (완전 오픈)
- **무엇을 하나**: 표적 추적·상태추정 프레임워크 — 운동/측정 모델, EKF/UKF·파티클 필터, 데이터연관(JPDA), 다중표적 추적, track 개시/삭제, OSPA·CLEAR MOT 평가. radar/passive sensor/simulator 모듈 공식 제공
- **표적 산란 처리**: n/a (추적 프레임워크)
- **검출체인?** 검출 이후의 **추적**(우리는 future work 로 미룬 단계)
- **우리 프로젝트 채택**: MEDIUM(추적 단계) — 바이스태틱은 custom nonlinear 측정모델 필요(ρ_b=R_T+R_R−L, f_D=(û_T+û_R)·v/λ). 우리 트래킹 도입 시 EKF/UKF 를 직접 안 짜고 Stone Soup 로

### 🔎직접확인  openEMS  —  채택도 판정: **MEDIUM**
- **저장소·출처**: github.com/thliebig/openEMS  ([1](https://github.com/thliebig/openEMS))
- **라이선스**: GPLv3
- **무엇을 하나**: 완전 전자기장(full-wave) EC-FDTD 솔버 — Python/MATLAB 인터페이스, **RCS 계산 튜토리얼(RCS_Sphere)** 제공. 3D Maxwell 직접 풂
- **표적 산란 처리**: full-wave RCS(가장 정확) — σ(f,az,el,pol) 룩업테이블 오프라인 생성
- **검출체인?** n/a (EM 솔버)
- **우리 프로젝트 채택**: MEDIUM(RCS 상위검증) — RadarSimPy 보다 **더 정확한** σ 오라클(단 λ/10 메싱이라 드론 전체는 매우 무겁다). 오프라인 drone_rcs.h5 룩업 생성용. 우리 SBR+PO/RadarSimPy 를 한 번 더 앵커

### 🔎직접확인  GNU Radio  —  채택도 판정: **MEDIUM**
- **저장소·출처**: github.com/gnuradio/gnuradio  ([1](https://github.com/gnuradio/gnuradio))
- **라이선스**: GPLv3
- **무엇을 하나**: SDR 신호처리 런타임 — Python flow graph + C++ 고속 블록. 오프라인 시뮬↔실측 **동일 I/Q 인터페이스**(SigMF/HDF5)
- **표적 산란 처리**: n/a
- **검출체인?** n/a (런타임/I-O)
- **우리 프로젝트 채택**: MEDIUM(실측 I/O 표준화) — Sionna 시뮬 I/Q 와 X410 실측 I/Q 를 **같은 포맷(SigMF)**으로 통일해 동일 처리코드가 둘 다 먹게. OpenISAC 과 함께 실측단계

### 🔎직접확인  NIST ISAC-PLM  —  채택도 판정: **LOW-MEDIUM**
- **저장소·출처**: github.com/wigig-tools/isac-plm (Steve Blandino, NIST)  ([1](https://github.com/wigig-tools/isac-plm))
- **라이선스**: NIST(US Gov)
- **무엇을 하나**: IEEE **802.11ay/bf** ISAC PHY 모델(MATLAB) — SC/OFDM, MU-MIMO, 동기·채널추정·CFO, **클러터제거·2D-CFAR·도플러·active/passive 센싱**, 바이스태틱 거실 센싱
- **표적 산란 처리**: c) 점표적
- **검출체인?** ✅ 2D-CFAR·range/velocity
- **우리 프로젝트 채택**: LOW-MEDIUM(WiFi 센싱 참조) — WiFi 패시브 센싱 처리구조 참조에 좋음. ⚠ 802.11ay/bf 는 **60GHz DMG** 라 우리 2.4/5GHz WiFi 와 대역·전파특성 다름. MATLAB+WLAN Toolbox 필요(코드 이식 아님)

### ⚠미검증단서  MATLAB 5G/Radar/Phased Array Toolboxes  —  채택도 판정: **MEDIUM**
- **저장소·출처**: MathWorks 공식 (유료)  ([1](https://www.mathworks.com/help/5g/ug/integrated-sensing-and-communication-using-5g-waveform.html))
- **라이선스**: 상용(학교 라이선스 흔함)
- **무엇을 하나**: 가장 완성형 ISAC 예제 — 5G NR PDSCH 전송→DM-RS 채널추정→이동 UAV range/angle→CFAR·추적. 바이스태틱 레이더 예제도 제공
- **표적 산란 처리**: c) RCS 상수 입력(phased.RadarTarget MeanRCS=)
- **검출체인?** ✅ 전부(CFAR·추적)
- **우리 프로젝트 채택**: MEDIUM(참조·1차 baseline) — 구조가 우리와 동일함을 확인하는 표준 레퍼런스. 파이썬 프로젝트라 코드 이식은 안 함

> **언제 쓰나.** **Stone Soup**=추적 단계(우리 future work — 바이스태틱 custom 측정모델 필요). **openEMS**=RCS 를 full-wave 로 한 번 더 앵커(느리므로 오프라인 룩업). **GNU Radio**=실측 I/Q 를 **SigMF** 로 시뮬과 통일. **ISAC-PLM**=WiFi 센싱 참조(단 60GHz DMG 대역). **MATLAB**=1차 baseline 참조.

---
## §3b. 채택도 LOW — 초기 제외

### ⚠미검증단서  OpenAirInterface (OAI)  —  채택도 판정: **LOW**
- **저장소·출처**: gitlab.eurecom.fr/oai/openairinterface5g  ([1](https://gitlab.eurecom.fr/oai/openairinterface5g))
- **라이선스**: OAI Public License
- **무엇을 하나**: 실제 5G NR RAN 스택(gNB/UE) — PDSCH·DM-RS·SRS·PRS, 실시간 IQ. 표준 완전호환 NR 파형
- **표적 산란 처리**: n/a (실 파형)
- **검출체인?** n/a
- **우리 프로젝트 채택**: LOW(초기 제외) — 난이도 최고(실 5G RAN·RF 캘리브·실시간). 우리 G1/G2/G3 표준 NR 을 실측할 최종단계에서만. 초기엔 OpenISAC 공통 OFDM baseline 로 충분

### 📄단일출처  ns3sionna  —  채택도 판정: **LOW — 네트워크/MAC 레벨. 우리 물리레이어 패시브 레이더와 층이 다름**
- **저장소·출처**: github.com/tkn-tub/ns3sionna (Zubow·Rösler·Dressler, TU Berlin) · arXiv:2412.20524  ([1](https://github.com/tkn-tub/ns3sionna) · [2](https://arxiv.org/abs/2412.20524))
- **라이선스**: 오픈소스
- **무엇을 하나**: ns-3 + Sionna RT 채널결합(SionnaPropagationLoss/Delay/Mobility 모델). CFR 태그로 CSI 노출
- **표적 산란 처리**: Sionna 확산(채널만)
- **검출체인?** 없음 — CSI 내보내기만, 레이더/RCS/CFAR 전무
- **우리 프로젝트 채택**: LOW — 네트워크/MAC 레벨. 우리 물리레이어 패시브 레이더와 층이 다름

### 📄단일출처  ms-van3t-rt  —  채택도 판정: **LOW — V2X 통신 전용**
- **저장소·출처**: github.com/robpegurri/ms-van3t-rt · arXiv:2501.00372 (PoliMi + NVIDIA J.Hoydis)  ([1](https://arxiv.org/pdf/2501.00372))
- **라이선스**: 오픈소스
- **무엇을 하나**: ns-3 V2X 풀스택 + Sionna RT 채널(디지털 네트워크 트윈)
- **표적 산란 처리**: Sionna 확산(채널만)
- **검출체인?** 없음 — ISAC/레이더/RCS/CFAR 전무
- **우리 프로젝트 채택**: LOW — V2X 통신 전용

> **왜 초기 제외인가.** **OAI**(실 5G RAN)는 난이도 최고 — 표준 NR 실측 최종단계에서만. **ns3sionna·ms-van3t** 는 ns-3 **네트워크층** 채널결합이라 CSI 만 내보내고 레이더 RCS/CFAR 이 없다. 지금 넣으면 연구 핵심보다 시스템 통합에 시간이 쏠린다.

In [ ]:
# 도구별 채택 판정 — prior_work.json 에서
import json
J = json.load(open('outputs/prior_work.json', encoding='utf-8'))
for t in J['tools']:
    tag = t['adopt'].split('(')[0].strip()
    print(f"{tag:8s} | {t['name']:26s} | {t['license'][:22]:22s} | {t['repo'][:40]}")

---
## §4. 정리 — 채택 계획

⭐ 갱신(사용자 이미지 심화 서베이 반영): 우리 손으로 만든 조각을 오픈소스로 단계적 대체 — **검출체인 ECA/CAF/CFAR → pyAPRiL**(GPLv3, DVB-T/FM 실검증; 앞서 '드롭인 없음'이라 한 것 정정) · **RCS → RadarSimPy(빠름)+openEMS(full-wave 앵커)** · **추적 → Stone Soup**(바이스태틱 custom 측정모델) · **실측 → OpenISAC+GNU Radio+X410**(SigMF 로 sim↔real I/Q 통일) · **파형/채널 → Sionna PHY 유지**(검증됨). 5G 실파형은 OAI, WiFi 센싱은 NIST ISAC-PLM 참조(60GHz 대역 주의).

요컨대 **직접 만든 조각을 계층별로 오픈소스로 대체**한다(‘검증 후 대체’, `OPENSOURCE.md`): **검출 ECA/CAF/CFAR → pyAPRiL**, **RCS → RadarSimPy(+openEMS 앵커)**, **추적 → Stone Soup**, **실측 → OpenISAC+GNU Radio+X410(SigMF)**. **파형/채널은 이미 Sionna PHY 로 검증**(report05)돼 유지. 이렇게 하면 우리가 검증 부담을 지던 코드를 실측·표준 검증된 도구가 나눠 져 **신뢰성이 오르고 반복 수작업이 준다** — 사용자 방침 그대로다.

> **다음** → [pw03 — 우리 방법의 위치와 선행 방법론 수용](pw03_positioning.ipynb).